# MM4250 sweeps

One notebook for every measurement on the switch: raw 1-port, calibrated 2-port, cal-vs-uncal, and the fridge e-cal.

**How to use it:** run the four **Setup** cells, then jump to the section you want and run just that section. Don't use *Run All*: it stops on purpose at the cell after Setup, so every sweep in here can't fire one after another.

| Section | What | VNA correction |
|---|---|---|
| **A** | 1-port S11 sweep | off (or on, your choice) |
| **B** | 2-port sweep | on: the saved cal set `CAL_SET` |
| **C** | cal vs. uncal, one channel | both, one after the other |
| **D** | fridge e-cal: calibrated S11 at each channel's connector | off, which the code enforces |
| **E** | browse and plot any runs | n/a |
| **F** | first cooldown: e-cal vs. a known short and load (see `FIRST_COOLDOWN.md`) | off, except the F1 bench reference |

**Each section sets the VNA up itself**: its own `setup_sweep(...)` and its own correction on/off line at the top. So it doesn't matter which section ran before it. The cal set in B/C is only valid at 1 MHz to 10 GHz, 10000 points, 1 kHz; D has to run raw.

**Files it needs**, in this folder: `vna_measure.py`, `sweep_db.py`, `read_db.py`, `plots.py`, `ecal.py`. For section D on the lab computer, also a copy of NIST's standard definitions: `ideals_3K/` (and `ideals_295K/` for room temperature). See `LAB_SETUP.md`.

**What it writes**, beside this notebook: every run goes into `mm4250_sweeps.db`. `Sweeps/...` Touchstone files are written only with `touchstone=True`, `export_touchstone(runs)` or `ecal.export_corrected(result)`, and `figures/...` only when a plot is saved.

## Setup

### Imports

Finds this folder and the `drivers/` folder above it (prints both), and imports everything the sections use. Also makes every run record which VNA cal set was active, as `vna_cal_set`.

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

# The drivers live in whichever repo root is above this notebook -- this
# folder if it's the mm4250-switch repo, or the framework repo if these
# files were copied into users/<name>/. Walk up until we find it.
try:
    here = Path(_dh[0])  # Jupyter sets _dh[0] to this notebook's directory
except NameError:
    here = Path.cwd()
if str(here) not in sys.path:
    sys.path.insert(0, str(here))       # vna_measure / sweep_db live here

root = here
while not (root / "drivers").is_dir() and root != root.parent:
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))       # drivers/ live here
print(f"notebook folder: {here}")
print(f"drivers from:    {root}")

from drivers.KeysightVNA_driver import KeysightP5004B
from drivers.MM4250_QCodes_driver import MM4250
from datetime import date

import vna_measure
from vna_measure import setup_sweep, sweep_settings, measure_s11, measure_2port
from sweep_db import run_oneport_sweep, run_twoport_sweep, run_ecal_set
from plots import plot_measurement, plot_sweep, summarize
from read_db import list_runs, load_run, export_touchstone
import ecal

# Record the active cal set's name in every run's metadata (vna_cal_set).
vna_measure.VNA_STATE_QUERIES["cal_set"] = "SENS:CORR:CSET:ACT? NAME"

### Connect

Creates `ksvna` (Keysight P5004B) and `switch` (MM4250). Connecting the switch **forces `ALL_OPEN`**: re-running this mid-session resets the switch. If the switch won't connect, another kernel still holds it. If `import hid` fails, see *Debugging* at the bottom.

In [ ]:
ksvna = KeysightP5004B("ksvna", "TCPIP0::QTSF-Measurement::hislip_PXI0_CHASSIS1_SLOT1_INDEX0::INSTR")
switch = MM4250("switch")

### Session

Set these once per session. Every section below files its runs under them. `temp_str` is a folder label; in the fridge, also pass the actual mixing-chamber reading to `run_ecal_set(mxc_temp_k=...)`.

In [ ]:
date_str = date.today().strftime("%Y%m%d")   # or type one, e.g. "20261015"
temp_str = "295K"            # "295K", "3K", "25mK", ...
switch_serials = "SN0077"    # the switch under test (e.g. "SN0077_SN0078" for two)
CAL_SET = "20260911_1MHz_10GHz_female"   # the VNA's saved 2-port cal, for sections B and C

print(f"{switch_serials} | {date_str} | {temp_str}")

### Stop here

No need to run. This cell exists so that *Run All* ends after Setup. From here, run a section by hand.

In [ ]:
raise SystemExit("Setup done. Now run one section below by hand.")

## Check the VNA

Saved cal sets, current sweep settings, whether correction is on (`1`) and which cal set is active. Worth a look before any section.

In [ ]:
print("Saved cal sets:", ksvna.ask("SENS:CORR:CSET:CAT? NAME").strip())
sweep_settings(vna=ksvna)
print("Correction on:", ksvna.ask("SENS:CORR:STAT?").strip())
print("Active cal set:", ksvna.ask("SENS:CORR:CSET:ACT? NAME").strip())

## Quick look (nothing saved)

One position, measured with whatever the VNA is currently set to. `measure_s11(3)` or `measure_s11(state="INTERNAL_LOAD")`; `measure_2port(3)` for all four S-parameters.

In [ ]:
freq, s11 = measure_s11(3, vna=ksvna, switch=switch)
summarize(freq, s11, title="RF3")
ax = plot_measurement(freq, s11, title="RF3")

# freq, data = measure_2port(3, vna=ksvna, switch=switch)
# ax = plot_measurement(freq, data, title="RF3", phase=True)

## A. 1-port sweep

S11 at each position in `positions`: channel numbers, state names (`"ALL_OPEN"`, `"INTERNAL_LOAD"`, `"INTERNAL_SHORT"`, `"RFC_RF1"`...), or a mix. Raw by default; to have the VNA correct it, activate `CAL_SET` instead (a 2-port cal covers port 1's S11 too), with the matching `setup_sweep` from section B.

In [ ]:
setup_sweep(start=1e9, stop=10e9, points=1001, if_bandwidth=1e3, power=-20, vna=ksvna)
ksvna.write('SENS:CORR:CSET:DEAC')            # raw

positions = [1, 2, 3, 4, 5, 6]
setup = None                                  # names the cabling, e.g. "RF3" -> folder RF3_uncal

runs = run_oneport_sweep(positions, date_str, temp_str, switch_serials, setup=setup,
                         vna=ksvna, switch=switch)      # touchstone=True for .s1p files too
ax = plot_sweep(runs)

In [ ]:
# Optional: S11 of a bare termination on the cable (no switch), saved straight to a file.
# Uncomment to use; rename the file for each standard/temperature.
# from read_db import save_touchstone
# freq, s11 = measure_s11(vna=ksvna)          # no channel = whatever is on the cable
# save_touchstone(freq, s11, "Sweeps/terminations/short_295K.s1p")

## B. 2-port sweep (VNA-calibrated)

S11, S12, S21, S22 at each position, with the VNA's cal set on. The sweep settings must be the ones the cal set was taken at. Measuring `[n, "ALL_OPEN"]` with port 2 on RF`n` gives isolation without touching a cable.

In [ ]:
setup_sweep(start=1e6, stop=10e9, points=10000, if_bandwidth=1e3, power=-20, vna=ksvna)   # must match CAL_SET
ksvna.write(f'SENS:CORR:CSET:ACT "{CAL_SET}",0')     # ,0 keeps the settings above

positions = [6, "ALL_OPEN"]
setup = "RF6"                                 # the cabling: VNA port 2 on RF6 -> folder RF6_cal

runs = run_twoport_sweep(positions, date_str, temp_str, switch_serials, setup=setup,
                         vna=ksvna, switch=switch)      # touchstone=True for .s2p files too
ax = plot_sweep(runs)

## C. Cal vs. uncal, one channel

VNA port 1 on RFC, port 2 on RF`n`. The same positions measured with the cal set on, then off, so the two can be compared directly. Change `n`, re-cable, run again.

In [ ]:
setup_sweep(start=1e6, stop=10e9, points=10000, if_bandwidth=1e3, power=-20, vna=ksvna)   # must match CAL_SET
n = 6
positions = [n, "ALL_OPEN", "INTERNAL_LOAD", "INTERNAL_SHORT"]

ksvna.write(f'SENS:CORR:CSET:ACT "{CAL_SET}",0')     # -> RF<n>_cal
runs_cal = run_twoport_sweep(positions, date_str, temp_str, switch_serials,
                             setup=f"RF{n}", vna=ksvna, switch=switch)

ksvna.write('SENS:CORR:CSET:DEAC')                   # -> RF<n>_uncal
runs_uncal = run_twoport_sweep(positions, date_str, temp_str, switch_serials,
                               setup=f"RF{n}", vna=ksvna, switch=switch)

runs = runs_cal + runs_uncal
print(f"cal runs {runs_cal}, uncal runs {runs_uncal}")
ax = plot_sweep(runs, sparam="S21")

## D. Fridge e-cal

Calibrated S11 at each RF channel's SMA connector, using the switch's internal open/short/load and NIST's definitions of them, so the whole fridge between the VNA and the switch is removed. **One set per temperature.** Take one at 3 K, and again at base.

**Wiring:** VNA port 1 must reach the switch's RFC port with enough return signal to measure a reflection, i.e. a cold coupler plus an amplifier on the way back up, not just one attenuated line.

**D1** measures one set: the standards, every channel, then the standards again (the drift check), `repeats` times. It refuses to run with VNA correction on, and it leaves the switch `ALL_OPEN`. Put whatever is screwed onto the RF connectors in `terminations` (e.g. `{2: "short", 5: "load"}`): each channel's run is tagged with it, and the D2 plot labels the channels with it. Don't change `setup_sweep` partway through a set.

Power: what reaches the switch is this minus the input line's attenuation. NIST had about -52 dBm at the switch. For less noise, lower `if_bandwidth` first. But a slower set gives the chain longer to drift, so check `drift` in D2 when you do.

In [ ]:
setup_sweep(start=1e6, stop=10e9, points=2001, if_bandwidth=1e3, power=-20, vna=ksvna)
ksvna.write('SENS:CORR:CSET:DEAC')            # e-cal wants raw data

cal = run_ecal_set(date_str, temp_str, switch_serials,
                   channels=[1, 2, 3, 4, 5, 6],
                   repeats=2,
                   mxc_temp_k=None,           # the mixing-chamber reading, e.g. 3.1
                   note="",                   # anything else worth remembering
                   terminations=None,         # what's on each RF connector, e.g. {2: "short", 5: "load"};
                                              # ports left out are recorded as "none" (nothing attached)
                   vna=ksvna, switch=switch)

**D2** corrects the set and shows it. Use NIST's **3K** definitions for anything cold, base included. NIST found they hold to 25 mK. Use **295K** only for a room-temperature set. `correct_set(None, ideals)` works on the most recent set in the database, even from a fresh kernel; `ecal.list_sets()` shows them all.

What to look at: `drift` (how far the raw standards moved during the set; NIST's same-temperature repeatability was a few 1e-2), `repeatability` (spread between repeats), then the plot. A short or open on a channel should read near 0 dB.

In [ ]:
ideals = ecal.find_ideals("3K", start=here)     # "295K" for a room-temperature set
result = ecal.correct_set(cal, ideals)          # or ecal.correct_set(None, ideals) for the latest set

ecal.drift(cal)
if len(result["per_repeat"]) > 1:
    ecal.repeatability(result)
axes = ecal.plot_ecal(result)                    # show_raw=True to overlay the uncorrected traces

# ecal.export_corrected(result)                 # .s1p per channel, if something outside needs files

## E. Browse and plot

`plot_sweep` reads from the database, so any runs from any session work: run ids, or a folder of Touchstone files. `list_runs()` shows what's there.

In [ ]:
for r in list_runs()[-10:]:
    print(r["run_id"], r["experiment"], r["name"])

# ax = plot_sweep([31, 39], sparam="S11")
# ax = plot_sweep("Sweeps/SN0077/20260925/295K/RF1_cal", sparam="S11",
#                 xlim=(0, 6), title="SN0077 RF1, 295 K", save=True)
# ax = plot_measurement(run=39, phase=True)
# export_touchstone([31, 32, 33])

## F. First cooldown: known short and load

A short and a load on two RF ports, e-cal sets warm and cold, scored against a bench measurement of the same short and load. `FIRST_COOLDOWN.md` has the why and what to expect. Run these in order **across the cooldown**, not in one go. Only F2, F3 and F4 move the switch; between F2 and F4 the fridge cools with every channel open.

**F0** What's on the ports, and the bench run ids once F1 has run. Re-run after a kernel restart.

In [ ]:
TERMS = {2: "short", 5: "load"}     # EDIT: channel -> what's screwed onto it; ports left out are bare
BENCH = {}                          # after F1, paste its printout here, e.g. {"short": 812, "load": 813}

**F1** Bench reference, 295 K, before the switch goes in: the short and load straight on VNA port 1's cable with `CAL_SET` on. No switch involved. Saved to the database.

In [ ]:
from sweep_db import _open_experiment, record_measurement
from vna_measure import instrument_state

setup_sweep(start=1e6, stop=10e9, points=10001, if_bandwidth=1e3, power=-20, vna=ksvna)  # must match CAL_SET
ksvna.write(f'SENS:CORR:CSET:ACT "{CAL_SET}",0')

exp, _ = _open_experiment("terminations", None, f"{date_str}_295K_bench_terminations")
for what in ("short", "load"):
    input(f"Put the {what} on VNA port 1's cable, then press Enter...")
    freq, s11 = measure_s11(vna=ksvna)          # no channel: the switch isn't involved
    ds = record_measurement(f"{what}_bench", freq, s11, exp, termination=what,
                            date_str=date_str, temp_str="295K",
                            **instrument_state(("S11",), vna=ksvna))
    BENCH[what] = ds.run_id
print(f"BENCH = {BENCH}    <- paste into F0")

**F2** Warm e-cal set in the fridge (295 K), wired as it will be cold, before pumpdown. Leaves the switch `ALL_OPEN`: nothing switches again until 3 K is stable. Score it with F5.

In [ ]:
setup_sweep(start=1e6, stop=10e9, points=2001, if_bandwidth=1e3, power=-20, vna=ksvna)
ksvna.write('SENS:CORR:CSET:DEAC')            # e-cal wants raw data

cal_warm = run_ecal_set(date_str, "295K", switch_serials, channels=[1, 2, 3, 4, 5, 6],
                        repeats=2, terminations=TERMS,
                        note="in fridge, warm, before pumpdown", vna=ksvna, switch=switch)
ecal.drift(cal_warm)

**F3** Stuck check, first thing at 3 K (stable). Every `ALL_OPEN` row should be tiny and every other row clearly bigger; compare rows with each other, not to a fixed number. A closed position about as small as `ALL_OPEN` = stuck open. An `ALL_OPEN` row that looks like the position before it = stuck closed. **If anything is stuck, stop**: warm up (switches recover at 30 to 80 K), see `FIRST_COOLDOWN.md` step 4.

In [ ]:
setup_sweep(start=1e6, stop=10e9, points=2001, if_bandwidth=1e3, power=-20, vna=ksvna)
ksvna.write('SENS:CORR:CSET:DEAC')

import numpy as np
SHORT_CH = next(ch for ch, t in TERMS.items() if t == "short")
LOAD_CH = next(ch for ch, t in TERMS.items() if t == "load")
seq = ["ALL_OPEN", "INTERNAL_SHORT", "ALL_OPEN", "INTERNAL_LOAD", "ALL_OPEN",
       SHORT_CH, "ALL_OPEN", LOAD_CH, "ALL_OPEN"]
runs = run_oneport_sweep(seq, date_str, "3K", switch_serials, setup="stuckcheck",
                         vna=ksvna, switch=switch)

ref = load_run(runs[0])[1]["S11"]
for pos, r in zip(seq, runs):
    g = load_run(r)[1]["S11"]
    print(f"{str(pos):>15}  run {r}:  median |S11 - first ALL_OPEN| = {np.median(np.abs(g - ref)):.3g}")

**F4** Cold e-cal set. At 3 K first, then again at base with `temp` set to the base label (e.g. `"25mK"`). Look at the drift printout: well above NIST's few 1e-2 means the chain moved during the set; take it again.

In [ ]:
temp = "3K"                                   # then the base label, e.g. "25mK"
setup_sweep(start=1e6, stop=10e9, points=2001, if_bandwidth=1e3, power=-20, vna=ksvna)
ksvna.write('SENS:CORR:CSET:DEAC')

cal_cold = run_ecal_set(date_str, temp, switch_serials, channels=[1, 2, 3, 4, 5, 6],
                        repeats=2,
                        mxc_temp_k=None,      # the reading of the plate the switch is on, e.g. 3.1
                        terminations=TERMS, note="", vna=ksvna, switch=switch)
ecal.drift(cal_cold)

**F5** Score it: each terminated port's calibrated S11, alone and against its bench run. Short: near 0 dB, and matching the bench in phase. Load: look at `|dS11|` against the bench, not dB or phase. Expected numbers are in `FIRST_COOLDOWN.md` step 6. The first cell only defines the function. In the second, comment out whichever set you haven't taken yet; from a fresh kernel, pass set id strings from `ecal.list_sets()` instead of `cal_warm` / `cal_cold`.

In [ ]:
import numpy as np

def check_terminations(result, bench=None, band=(0.3e9, 8e9)):
    """Calibrated S11 of each terminated port, alone and against its bench reference."""
    f = result["freq"]
    keep = (f >= band[0]) & (f <= band[1])
    for ch, what in sorted(result["terminations"].items()):
        if ch not in result["ports"]:
            continue
        g = result["ports"][ch][keep]
        db = 20 * np.log10(np.abs(g))
        print(f"RF{ch} ({what}): |S11| median {np.median(db):+.2f} dB, "
              f"range {db.min():+.2f} to {db.max():+.2f} dB")
        run = (bench or {}).get(what)
        if run is None:
            continue
        fr, data = load_run(run)
        s = data["S11"]
        ref = np.interp(f[keep], fr, s.real) + 1j * np.interp(f[keep], fr, s.imag)
        dv = np.abs(g - ref)
        print(f"    vs bench run {run}: |dS11| median {np.median(dv):.3g}, max {dv.max():.3g}")
        if np.median(np.abs(ref)) > 0.5:          # magnitude/phase only mean something for a reflective standard
            ddb = db - 20 * np.log10(np.abs(ref))
            dph = np.angle(g / ref, deg=True)
            print(f"    {np.median(ddb):+.2f} dB median (worst {ddb[np.argmax(np.abs(ddb))]:+.2f}), "
                  f"{np.median(dph):+.1f} deg median (worst {dph[np.argmax(np.abs(dph))]:+.1f})")

In [ ]:
result_warm = ecal.correct_set(cal_warm, ecal.find_ideals("295K", start=here))
check_terminations(result_warm, BENCH)

result_cold = ecal.correct_set(cal_cold, ecal.find_ideals("3K", start=here))   # 3K definitions for base too
check_terminations(result_cold, BENCH)
axes = ecal.plot_ecal(result_cold, show_raw=True)

## Close

Backs up the database to `db_backups/` (nothing is written if no runs are new since the last backup), then turns the VNA output off, opens every switch channel and releases both instruments. Run it before another notebook needs the switch, because the MM4250's USB connection is exclusive.

In [ ]:
from read_db import backup_db
backup_db()          # snapshot to db_backups/ beside the .db; skipped if nothing new

ksvna.output(False)
switch.open_all()
ksvna.close()
switch.close()

## Debugging: hidapi

Only needed if `import hid` fails. See the hidapi section of `LAB_SETUP.md`.

In [ ]:
#checking hidapi properly active
import sys, subprocess, importlib
r = subprocess.run([sys.executable, "-m", "pip", "install", "hidapi"], capture_output=True, text=True)
print(r.stdout[-1500:])
print(r.stderr[-1500:])

importlib.invalidate_caches()
try:
    import hid
    print("OK:", hid.__file__)
except Exception as e:
    print("FAILED:", type(e).__name__, e)
print("python:", sys.executable)

In [ ]:
#more hid stuff
import importlib, importlib.util
importlib.invalidate_caches()
print(importlib.util.find_spec("hid"))